In [1]:
import matplotlib.pyplot as plt
import json
from graphviz import Digraph
from PIL import Image

## Combine Data

In [2]:
def read_json_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return json.load(file)

def add_not_to_exc(exc_data):
    return {
        "NOT": {
            "left": exc_data
        }
    }
def combine_criteria(inc_data, exc_data):
    return {
        "AND": {
            "left": inc_data,
            "right": exc_data
        }
    }

In [3]:
inc_file = "data/NCT03860402_inc.json"
exc_file = "data/NCT03860402_exc.json"
inc_data = read_json_file(inc_file)
exc_data = read_json_file(exc_file)
exc_data_with_not = add_not_to_exc(exc_data)
combined_data = combine_criteria(inc_data, exc_data_with_not)

In [4]:
def parse_logic_to_tree(logic, graph, parent=None, node_id=0):
    if 'raw_text' in logic:
        node_label = logic['raw_text']
        graph.node(str(node_id), label=node_label, shape='box')
        if parent is not None:
            graph.edge(parent, str(node_id))
        return node_id
    for key in logic:
        if key in ('AND', 'OR', 'NOT'):
            operator = key
            node_label = operator
            current_node_id = node_id
            color = 'lightblue' if operator == 'AND' else 'lightgreen' if operator == 'OR' else 'lightcoral'
            graph.node(str(current_node_id), label=node_label, color=color, fontcolor='black', style='filled', fillcolor=color)
            if parent is not None:
                graph.edge(parent, str(current_node_id))
            operands = logic[key]
            node_id += 1
            if 'left' in operands:
                left_id = parse_logic_to_tree(operands['left'], graph, str(current_node_id), node_id)
                node_id = left_id + 1
            if 'right' in operands:
                right_id = parse_logic_to_tree(operands['right'], graph, str(current_node_id), node_id)
                node_id = right_id + 1
            if operator == 'NOT':
                node_id += 1
    return node_id

def visualize_logic_tree(logic):
    graph = Digraph(format='png', encoding='utf-8')
    parse_logic_to_tree(logic, graph)
    return graph

def plot_and_save_graph(logic, graph_title, output_filename):
    graph = visualize_logic_tree(logic)
    graph.render(filename='temp', view=False, cleanup=True)
    image = Image.open('temp.png')
    plt.figure(figsize=(20, 15))
    plt.imshow(image)
    plt.axis('off')
    plt.title(graph_title)
    plt.savefig(output_filename)
    plt.close()

In [5]:
plot_and_save_graph(combined_data, "", "logic_tree.png")

#### Merge EC with Entities

In [6]:
def parse_logic_to_tree(logic, graph, parent=None, node_id=0):
    if 'raw_text' in logic:
        node_label = logic['raw_text']
        graph.node(str(node_id), label=node_label, shape='box', style='filled', fillcolor='lightyellow')
        if parent is not None:
            graph.edge(parent, str(node_id))
        entity_id = node_id + 1
        for key, values in logic.items():
            if key not in ['raw_text', 'AND', 'OR', 'NOT']:
                for value in values:
                    graph.node(f"{entity_id}", label=f"{key}: {value}", shape='ellipse', style='filled', fillcolor='lightcyan')
                    graph.edge(str(node_id), f"{entity_id}")
                    entity_id += 1
        return entity_id - 1
    for key in logic:
        if key in ('AND', 'OR', 'NOT'):
            operator = key
            node_label = operator
            current_node_id = node_id
            color = 'lightblue' if operator == 'AND' else 'lightgreen' if operator == 'OR' else 'lightcoral'
            graph.node(str(current_node_id), label=node_label, shape='circle', style='filled', fillcolor=color)
            if parent is not None:
                graph.edge(parent, str(current_node_id))
            operands = logic[key]
            node_id += 1
            if 'left' in operands:
                left_id = parse_logic_to_tree(operands['left'], graph, str(current_node_id), node_id)
                node_id = left_id + 1
            if 'right' in operands:
                right_id = parse_logic_to_tree(operands['right'], graph, str(current_node_id), node_id)
                node_id = right_id + 1
            if operator == 'NOT':
                node_id += 1
    return node_id

def visualize_logic_tree(logic):
    graph = Digraph(format='png', engine='dot', encoding='utf-8')
    graph.attr(rankdir='TB', size='8,8', dpi='300')
    graph.attr('node', fontname='Arial', fontsize='10')
    graph.attr('edge', fontname='Arial', fontsize='8')
    parse_logic_to_tree(logic, graph)
    return graph

def plot_and_save_graph(logic, output_filename):
    graph = visualize_logic_tree(logic)
    graph.render(filename='temp', view=False, cleanup=True)
    image = Image.open('temp.png')
    plt.figure(figsize=(60, 30))
    plt.imshow(image)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(output_filename, bbox_inches='tight')
    plt.close()


In [7]:
exc_data_with_not = add_not_to_exc(exc_data)
combined_data = combine_criteria(inc_data, exc_data_with_not)
plot_and_save_graph(combined_data, "combined_tree_ent.png")

# INC and EXC To PDF

In [8]:
def read_json_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return json.load(file)

def parse_logic_to_tree(logic, graph, parent=None, node_id=0):
    if 'raw_text' in logic:
        node_label = logic['raw_text']
        graph.node(str(node_id), label=node_label, shape='box', style='filled', fillcolor='lightyellow',
                   width='4', height='1.8', fontsize='40', fontweight="bold")  # Schriftgröße auf 36
        if parent is not None:
            graph.edge(parent, str(node_id))
        entity_id = node_id + 1
        for key, values in logic.items():
            if key not in ['raw_text', 'AND', 'OR', 'NOT']:
                for value in values:
                    graph.node(f"{entity_id}", label=f"{key}: {value}", shape='ellipse', style='filled',
                               fillcolor='lightcyan', width='3', height='1.5', fontsize='39')  # Schriftgröße auf 32
                    graph.edge(str(node_id), f"{entity_id}")
                    entity_id += 1
        return entity_id - 1
    for key in logic:
        if key in ('AND', 'OR', 'NOT'):
            operator = key
            node_label = operator
            current_node_id = node_id
            color = 'lightblue' if operator == 'AND' else 'lightgreen' if operator == 'OR' else 'lightcoral'
            graph.node(str(current_node_id), label=node_label, shape='circle', style='filled',
                       fillcolor=color, width='2', height='2', fontsize='45')  # Schriftgröße auf 40
            if parent is not None:
                graph.edge(parent, str(current_node_id))
            operands = logic[key]
            node_id += 1
            if 'left' in operands:
                left_id = parse_logic_to_tree(operands['left'], graph, str(current_node_id), node_id)
                node_id = left_id + 1
            if 'right' in operands:
                right_id = parse_logic_to_tree(operands['right'], graph, str(current_node_id), node_id)
                node_id = right_id + 1
            if operator == 'NOT':
                node_id += 1
    return node_id

def visualize_logic_tree(logic):
    graph = Digraph(format='pdf', engine='dot')
    graph.attr(rankdir='TB')
    graph.attr(size='16.5,66.0')
    graph.attr(nodesep='2.2', ranksep='10.2')
    graph.attr('node', fontsize='32', fontweight='bold')  
    graph.attr('edge', fontsize='28', fontweight='bold', penwidth='4.0') 
    graph.attr(margin='0.5')
    graph.attr()
    parse_logic_to_tree(logic, graph)
    return graph

def plot_and_save_graph(logic, output_filename):
    graph = visualize_logic_tree(logic)
    graph.render(filename=output_filename, cleanup=True)

In [9]:
plot_and_save_graph(inc_data, "inc_tree_ent")
plot_and_save_graph(exc_data, "exc_tree_ent")

In [12]:
import json
from graphviz import Digraph
import uuid
import os

def read_json_file(filename):
    with open(filename, 'r') as f:
        return json.load(f)

def visualize_ast(json_data, output_file="ast_tree"):
    if isinstance(json_data, str):
        data = json.loads(json_data)
    else:
        data = json_data
    dot = Digraph(comment='AST Visualization')
    dot.attr(rankdir='TB')
    dot.attr('edge', arrowhead='normal', arrowsize='1')

    def create_entity_html(entities):
        html = ''
        for key, values in entities.items():
            if key not in ['raw_text'] and isinstance(values, list):
                html += f'<TR><TD ALIGN="LEFT"><FONT COLOR="#666666">{key}: {", ".join(values)}</FONT></TD></TR>'
        return html

    def add_node(data, parent_id=None):
        current_id = str(uuid.uuid4())
        if isinstance(data, dict) and any(op in data for op in ['AND', 'OR', 'NOT']):
            operator = next(op for op in ['AND', 'OR', 'NOT'] if op in data)
            color = {
                'AND': 'lightblue',
                'OR': 'lightgreen',
                'NOT': 'lightcoral'
            }[operator]

            # Create operator node with rounded shape and color
            dot.node(current_id, operator,
                     shape='circle',
                     style='filled',
                     fillcolor=color,
                     fontcolor='black')

            # Ensure that each node only has one outgoing edge per child
            if operator == 'NOT':
                # NOT operator has only one child (left)
                child_data = data[operator]['left']
                child_id = add_node(child_data)
                dot.edge(current_id, child_id)
            else:
                # AND/OR operators have left and right children
                left_id = add_node(data[operator]['left'])
                right_id = add_node(data[operator]['right'])
                dot.edge(current_id, left_id)
                dot.edge(current_id, right_id)

        else:
            if isinstance(data, dict) and 'raw_text' in data:
                label = f'''<<TABLE BORDER="0" CELLBORDER="0" CELLSPACING="0">
                    <TR><TD><FONT FACE="bold">{data['raw_text']}</FONT></TD></TR>
                    {create_entity_html(data)}
                    </TABLE>>'''
                dot.node(current_id, label, shape='box')
            else:
                dot.node(current_id, str(data), shape='box')

        if parent_id:
            dot.edge(parent_id, current_id)

        return current_id

    add_node(data)

    dot.render(output_file, view=False, format='pdf', cleanup=True)
    output_dir = os.path.dirname(output_file) if os.path.dirname(output_file) else '.'
    pdf_path = os.path.join(output_dir, f"{os.path.basename(output_file)}.pdf")

    return pdf_path


inc_file = "data/NCT03860402_inc.json"
exc_file = "data/NCT03860402_exc.json"

inc_data = read_json_file(inc_file)
exc_data = read_json_file(exc_file)

# Visualize and save both trees
visualize_ast(exc_data, "exc_tree")
visualize_ast(inc_data, "inc_tree_ent") 

'.\\inc_tree_ent.pdf'